In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import requests

In [2]:
# load.env-file
load_dotenv()

schema = "sql_cities"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'


In [3]:

url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.31","lon":"13.24","radiusKm":"50","limit":"50","withFlightInfoOnly":"true"}

headers = {
    "Accept": "application/json, application/xml",
    "X-RapidAPI-Key": "x_rapid_api_key",
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'message': 'You are not subscribed to this API.'}


In [4]:
pd.json_normalize(response.json()['items'])

KeyError: 'items'

In [ ]:
def get_airports(latitudes, longitudes):
    # API headers
    headers = {
        "x-rapidapi-key": "x_rapid_api_key",
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    # DataFrame to store results
    all_airports = []

    for lat, lon in zip(latitudes, longitudes):
    # Construct the URL with the latitude and longitude
        url = url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"lat":lat,"lon":lon,"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)

    return pd.concat(all_airports, ignore_index=True)

In [ ]:
# coordinates for Berlin, Paris, London
latitudes = [52.5200, 48.8567, 51.5072]
longitudes = [13.4050, 2.3522, -0.1275]

get_airports(latitudes, longitudes)

###### **Challenge:** Arrivals information
Using what you have been shown above, plus the skills you've learnt in the last couple of days:
1. In `AeroDataBox API` use the `Flight API` > `FIDS/Schedules: Airport departures and arrivals (by time range)` section
2. Fill out the parameters in the middle third and then copy the `python: requests` code from the right hand third
3. Explore the data you get back. What would be useful in your DataFrame and what can be excluded? Remember Gans wants to know about when people are arriving in the city
4. Make a DataFrame from the information you see as important
5. Condense everything you did above into a function that can take a list of ICAO codes as an input, and as an output gives you a DataFrame with the information for *tomorrows arrivals*

In [ ]:
#flight request
code = "EDDB"
from_local = "2026-10-22T10:00"
to_local = "2026-10-22T12:00"

url = (
    f"https://aerodatabox.p.rapidapi.com/flights/airports/"
    f"icao/{code}/{from_local}/{to_local}"
)

querystring = {
    "direction": "Arrival",
    "withLeg": "true",
    "withCancelled": "false",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false",
    "withLocation": "true"
}

headers = {
    "Accept": "application/json",
    "X-RapidAPI-Key": "x_rapid_api_key",
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

try:
    response = requests.get(
        url,
        headers=headers,
        params=querystring,
        timeout=30
    )

    print("HTTP-Status:", response.status_code)
    print("Antwort:", response.text)

    response.raise_for_status()
    data = response.json()
    print(data)

except requests.exceptions.RequestException as e:
    print("HTTP-Fehler:", e)

except ValueError as e:
    print("Fehler beim Lesen der JSON-Antwort:", e)

pd.json_normalize(response.json()["arrivals"])

In [ ]:
df = pd.json_normalize(flights)
display(df.head())